In [ ]:
## Fase 2 — Datos sintéticos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

N_POINTS = 512
N_FAMILIES = 4
N_SAMPLES = 800

def generar_continuo(x, pendiente, nivel):
    """Simula el continuo estelar: una curva suave que sube o baja."""
    return nivel + pendiente * (x - x.mean()) / x.std()

def generar_lineas_emision(x, posiciones, amplitudes, ancho=3):
    """Simula líneas de emisión: picos gaussianos angostos en posiciones fijas."""
    lineas = np.zeros_like(x, dtype=float)
    for pos, amp in zip(posiciones, amplitudes):
        lineas += amp * np.exp(-0.5 * ((x - pos) / ancho) ** 2)
    return lineas

x = np.arange(N_POINTS)

# posiciones fijas de "líneas de emisión" (iguales para todas las familias,
# como en un espectro real: la posición de una línea no cambia, su amplitud sí)
posiciones_lineas = [100, 220, 340, 460]

# parámetros por familia: (pendiente_continuo, nivel_continuo, amplitudes_lineas)
parametros_familias = [
    (-0.3, 1.0, [0.1, 0.1, 0.1, 0.1]),   # "pasiva": continuo rojo, líneas débiles
    ( 0.4, 1.0, [2.0, 1.5, 1.8, 2.2]),   # "formando estrellas": continuo azul, líneas fuertes
    ( 0.0, 1.0, [0.8, 0.6, 0.7, 0.9]),   # "intermedia"
    (-0.1, 0.7, [0.3, 3.0, 0.3, 0.3]),   # "línea única dominante" (ej. tipo AGN)
]

espectros = []
etiquetas_familia = []  # solo para verificar resultados, el modelo no lo ve

muestras_por_familia = N_SAMPLES // N_FAMILIES
for familia_id, (pendiente, nivel, amps) in enumerate(parametros_familias):
    for _ in range(muestras_por_familia):
        # ruido en los parámetros para que no todas las muestras de una familia sean idénticas
        pendiente_ruido = pendiente + np.random.normal(0, 0.05)
        amps_ruido = [a + np.random.normal(0, 0.1) for a in amps]

        continuo = generar_continuo(x, pendiente_ruido, nivel)
        lineas = generar_lineas_emision(x, posiciones_lineas, amps_ruido)
        ruido_medicion = np.random.normal(0, 0.03, N_POINTS)

        espectro = continuo + lineas + ruido_medicion
        espectros.append(espectro)
        etiquetas_familia.append(familia_id)

espectros = np.array(espectros, dtype=np.float32)   # shape: (800, 512)
etiquetas_familia = np.array(etiquetas_familia)

print('shape de espectros:', espectros.shape)

# inspección visual — un ejemplo de cada familia
fig, ax = plt.subplots(N_FAMILIES, 1, figsize=(8, 8), sharex=True)
for familia_id in range(N_FAMILIES):
    idx = np.where(etiquetas_familia == familia_id)[0][0]
    ax[familia_id].plot(espectros[idx])
    ax[familia_id].set_title(f'Ejemplo familia {familia_id}')
plt.tight_layout()
plt.show()

In [ ]:
# Fase 2 - Normalización 

In [ ]:
ventana_continuo = slice(0, 60)  # región sin líneas, al inicio del espectro

for i in range(len(espectros)):
    nivel_medio = espectros[i][ventana_continuo].mean()
    espectros[i] = espectros[i] / nivel_medio

In [ ]:
# Fase 2 - División train/test set

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_fam_train, y_fam_test = train_test_split(
    espectros, etiquetas_familia, test_size=0.2, random_state=0, stratify=etiquetas_familia
)
print('train:', x_train.shape, ' test:', x_test.shape)

In [ ]:
# Fase 3 - Autoencoder 1D

In [ ]:
from keras.layers import Input, Dense, Conv1D, MaxPooling1D, UpSampling1D, Reshape, Flatten, LeakyReLU
from keras.models import Model
from keras import backend as K

ALPHA = 0.1

def CNN_1DAE(input_dims, latent_dims=16):
    # ------------------- Encoder --------------
    input_layer = Input(shape=(input_dims,), name='input')
    reshaped_layer = Reshape((input_dims, 1))(input_layer)

    x = Conv1D(32, 6, strides=1, padding='same')(reshaped_layer)
    x = LeakyReLU(alpha=ALPHA)(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(32, 6, strides=1, padding='same')(x)
    x = LeakyReLU(alpha=ALPHA)(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(32, 6, strides=1, padding='same')(x)
    x = LeakyReLU(alpha=ALPHA)(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(32, 6, strides=1, padding='same')(x)
    x = LeakyReLU(alpha=ALPHA)(x)
    x = MaxPooling1D(pool_size=2)(x)

    shape_before_flattening = x.shape[1:] 
    dims, fmaps = shape_before_flattening
    num_neurons = dims * fmaps

    x = Flatten()(x)
    x = Dense(latent_dims)(x)
    encoded = LeakyReLU(alpha=ALPHA, name='z')(x)

    encoder = Model(input_layer, encoded, name='encoder')

    # ------------------ Decoder --------------------
    x = Dense(num_neurons, name='decoder_11')(encoded)
    x = LeakyReLU(alpha=ALPHA, name='decoder_11a')(x)
    x = Reshape(shape_before_flattening, name='decoder_10')(x)

    x = UpSampling1D(2, name='decoder_9')(x)
    x = Conv1D(32, 6, padding='same', name='decoder_8')(x)
    x = LeakyReLU(alpha=ALPHA, name='decoder_8a')(x)

    x = UpSampling1D(2, name='decoder_7')(x)
    x = Conv1D(32, 6, padding='same', name='decoder_6')(x)
    x = LeakyReLU(alpha=ALPHA, name='decoder_6a')(x)

    x = UpSampling1D(2, name='decoder_5')(x)
    x = Conv1D(32, 6, padding='same', name='decoder_4')(x)
    x = LeakyReLU(alpha=ALPHA, name='decoder_4a')(x)

    x = UpSampling1D(2, name='decoder_3')(x)
    x = Conv1D(32, 6, padding='same', name='decoder_2')(x)
    x = LeakyReLU(alpha=ALPHA, name='decoder_2a')(x)

    x = Conv1D(1, 1, activation='linear', padding='same', name='decoder_1')(x)
    decoded = Flatten(name='decoder_0')(x)

    autoencoder = Model(input_layer, decoded, name='autoencoder')

    # ----------------- Decoder standalone ------------------
    encoded_input = Input(shape=(latent_dims,))
    d = autoencoder.get_layer('decoder_11')(encoded_input)
    d = autoencoder.get_layer('decoder_11a')(d)
    d = autoencoder.get_layer('decoder_10')(d)
    d = autoencoder.get_layer('decoder_9')(d)
    d = autoencoder.get_layer('decoder_8')(d)
    d = autoencoder.get_layer('decoder_8a')(d)
    d = autoencoder.get_layer('decoder_7')(d)
    d = autoencoder.get_layer('decoder_6')(d)
    d = autoencoder.get_layer('decoder_6a')(d)
    d = autoencoder.get_layer('decoder_5')(d)
    d = autoencoder.get_layer('decoder_4')(d)
    d = autoencoder.get_layer('decoder_4a')(d)
    d = autoencoder.get_layer('decoder_3')(d)
    d = autoencoder.get_layer('decoder_2')(d)
    d = autoencoder.get_layer('decoder_2a')(d)
    d = autoencoder.get_layer('decoder_1')(d)
    d = autoencoder.get_layer('decoder_0')(d)

    decoder = Model(inputs=encoded_input, outputs=d, name='decoder')

    print(autoencoder.summary())
    return (autoencoder, encoder, decoder)

In [ ]:
autoencoder, encoder, decoder = CNN_1DAE(input_dims=N_POINTS, latent_dims=16)

In [ ]:
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(x_train, x_train, epochs=20, batch_size=32)

In [ ]:
# Fase 4 - SOMLayer

In [ ]:
import tensorflow as tf
from keras.layers import Layer
from keras.layers import InputSpec

class SOMLayer(Layer):
    """
    Self-Organizing Map layer con topología rectangular
    """

    def __init__(self, map_size, prototypes=None, **kwargs):
        super(SOMLayer, self).__init__(**kwargs)
        self.map_size = map_size
        self.n_prototypes = map_size[0] * map_size[1]
        self.initial_prototypes = prototypes
        self.input_spec = InputSpec(ndim=2)

    def build(self, input_shape):
        input_dim = input_shape[1]
        self.input_spec = InputSpec(dtype=tf.float32, shape=(None, input_dim))
        self.prototypes = self.add_weight(
            shape=(self.n_prototypes, input_dim),
            initializer='glorot_uniform',
            name='prototypes'
        )
        if self.initial_prototypes is not None:
            self.set_weights(self.initial_prototypes)
            del self.initial_prototypes
        self.built = True

    def call(self, inputs, **kwargs):
        d = tf.reduce_sum(
            tf.square(tf.expand_dims(inputs, axis=1) - self.prototypes),
            axis=2
        )
        return d

    def compute_output_shape(self, input_shape):
        return input_shape[0], self.n_prototypes

    def get_config(self):
        config = {'map_size': self.map_size}
        base_config = super(SOMLayer, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

In [ ]:
# Fase 4 - Distancia que calcula SOMLayer (comprobación)

In [ ]:
import numpy as np

# simula un batch de 4 "codificaciones latentes" de 16 dimensiones
# (como si vinieran del encoder, pero acá estan inventadas a mano)
np.random.seed(0)
fake_latent_batch = np.random.rand(4, 16).astype(np.float32)

# crea la capa para un mapa chico de 5x5 = 25 prototipos
som_test = SOMLayer(map_size=(5, 5))

# hay que "construirla" pasándole datos, para que sepa el input_dim (16)
distances = som_test(fake_latent_batch)

print('Forma de las distancias:', distances.shape)   # esperado: (4, 25)
print(distances.numpy())

In [ ]:
bmu_indices = np.argmin(distances.numpy(), axis=1)
print('BMU de cada muestra:', bmu_indices)

In [ ]:
print(som_test.get_weights()[0].shape)

In [ ]:
# Fase 5 - DESOM

In [ ]:
def som_loss(weights, distances):
    """Pérdida SOM: suma pesada de distancias, promediada sobre el batch"""
    return tf.reduce_mean(tf.reduce_sum(weights * distances, axis=1))

def kmeans_loss(y_pred, distances):
    """Solo para diagnóstico/logging, no se usa para entrenar"""
    return np.mean([distances[i, y_pred[i]] for i in range(len(y_pred))])

In [ ]:
class DESOM:
    def __init__(self, input_dims, map_size, latent):
        self.input_dims = input_dims
        self.map_size = map_size
        self.n_prototypes = map_size[0] * map_size[1]
        self.latent = latent

    def initialize(self):
        self.autoencoder, self.encoder, self.decoder = CNN_1DAE(
            input_dims=self.input_dims, latent_dims=self.latent
        )
        som_layer = SOMLayer(self.map_size, name='SOM')(self.encoder.output)
        self.model = Model(inputs=self.autoencoder.input,
                            outputs=[self.autoencoder.output, som_layer])

    @property
    def prototypes(self):
        return self.model.get_layer(name='SOM').get_weights()[0]

    def compile(self, gamma, optimizer):
        self.model.compile(
            loss={'decoder_0': 'mse', 'SOM': som_loss},
            loss_weights=[1, gamma],
            optimizer=optimizer
        )

    def init_som_weights(self, X):
        sample = X[np.random.choice(X.shape[0], size=self.n_prototypes, replace=False)]
        encoded_sample = self.encode(sample)
        self.model.get_layer(name='SOM').set_weights([encoded_sample])

    def encode(self, x):
        return self.encoder.predict(x, verbose=0)

    def decode(self, x):
        return self.decoder.predict(x, verbose=0)

    def map_dist(self, y_pred):
        labels = np.arange(self.n_prototypes)
        tmp = np.expand_dims(y_pred, axis=1)
        d_row = np.abs(tmp - labels) // self.map_size[1]
        d_col = np.abs(tmp % self.map_size[1] - labels % self.map_size[1])
        return d_row + d_col

    def neighborhood_function(self, x, T):
        return np.exp(-(x**2) / (T**2))

In [ ]:
def fit_desom(desom, X_train, iterations=2000, som_iterations=1500,
              eval_interval=50, batch_size=32, Tmax=10, Tmin=0.1):

    index = 0
    logs = []

    for ite in range(iterations):
        # 1. armar el batch (con vuelta al principio si se acaba el dataset)
        if (index + 1) * batch_size >= X_train.shape[0]:
            X_batch = X_train[index * batch_size:]
            index = 0
        else:
            X_batch = X_train[index * batch_size:(index + 1) * batch_size]
            index += 1

        # 2. con los pesos ACTUALES de la red, ver dónde cae cada muestra
        _, d = desom.model.predict(X_batch, verbose=0)
        y_pred = d.argmin(axis=1)   # el BMU de cada muestra, ahora mismo

        # 3. enfriar la temperatura (el "annealing")
        if ite < som_iterations:
            T = Tmax * (Tmin / Tmax) ** (ite / (som_iterations - 1))

        # 4. calcular el peso de vecindad para cada muestra y cada prototipo
        w_batch = desom.neighborhood_function(desom.map_dist(y_pred), T)

        # 5. un paso de entrenamiento con backpropagation
        loss = desom.model.train_on_batch(X_batch, [X_batch, w_batch])

        if ite % eval_interval == 0:
            print(f'iter {ite:5d}  T={T:.3f}  L={loss[0]:.4f}  Lr={loss[1]:.4f}  Lsom={loss[2]:.4f}')
            logs.append({'iter': ite, 'T': T, 'L': loss[0], 'Lr': loss[1], 'Lsom': loss[2]})

    return logs

In [ ]:
from keras import optimizers

som = DESOM(input_dims=N_POINTS, map_size=(5, 5), latent=16)
som.initialize()
som.compile(gamma=5e-4, optimizer=optimizers.Adam(1e-3))
som.init_som_weights(x_train)

logs = fit_desom(som, x_train, iterations=2000, som_iterations=1500,
                  batch_size=32, Tmax=10, Tmin=0.1)